<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/BasicHarness4_MyCLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent: MyCLi

## ส่วนที่ 1: การกำหนดเครื่องมือ (Tool Definitions)
ในส่วนนี้เราจะนำเข้าไลบรารีที่จำเป็น และสร้างฟังก์ชันเครื่องมือ (Tools) ต่างๆ ที่เอเจนต์สามารถเรียกใช้งานได้ เช่น:
- `calc`: สำหรับคำนวณคณิตศาสตร์อย่างปลอดภัย
- `read_file`: สำหรับอ่านไฟล์ในไดเรกทอรีทำงาน
- `write_file`: สำหรับเขียนหรือแก้ไขไฟล์
- `run_terminal_command`: สำหรับรันคำสั่ง Linux Terminal
รวมถึงการกำหนด `SCHEMAS` เพื่อบอกให้โมเดล AI ทราบว่ามีเครื่องมืออะไรให้ใช้งานบ้างและใช้งานอย่างไร

In [ ]:
"""  Part 1 - Tool Definitions"""
import json  # Used for parsing and formatting JSON payloads for tool arguments
import re  # Used for secure input validation of arithmetic expressions
from pathlib import Path  # Used for safe, cross-platform file path resolution
import subprocess  # Used to execute terminal commands

def calc(expression: str) -> str:
    """Evaluate an arithmetic expression securely by limiting allowed characters."""
    # Only allow numbers, spaces, parentheses, and standard arithmetic operators
    if not re.match(r'^[0-9+\-*/().\s]+$', expression):
        raise ValueError("only numbers and + - * / are allowed")
    try:
        # Safe to evaluate since we restricted character set completely
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        raise ValueError(f"Invalid math expression: {e}")

def read_file(path: str) -> str:
    """Read a local file safely within the working directory."""
    p = Path(path).resolve()
    if not p.is_relative_to(Path.cwd().resolve()):
        raise PermissionError("path is outside the working directory")
    return p.read_text()[:8000]

def write_file(path: str, content: str) -> str:
    """Write content to a local file safely within the working directory."""
    p = Path(path).resolve()
    if not p.is_relative_to(Path.cwd().resolve()):
        raise PermissionError("path is outside the working directory")
    p.write_text(content)
    return f"Successfully wrote to {path}"

def run_terminal_command(command: str) -> str:
    """Run a Linux bash command in the local environment and return output."""
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )
        output = ""
        if result.stdout:
            output += f"Output:\n{result.stdout}\n"
        if result.stderr:
            output += f"Error/Stderr:\n{result.stderr}\n"
        if not output:
            output = f"Command completed with exit code: {result.returncode}"
        return output
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds."
    except Exception as e:
        return f"Error executing command: {e}"

# Tool schemas to pass to the model
SCHEMAS = [
    {"type": "function", "function": {
        "name": "calc", "description": "Evaluate an arithmetic expression like '12 * (3 + 4)'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a text file in the working directory.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "write_file", "description": "Write or overwrite a text file in the working directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"},
            "content": {"type": "string"}
        },
                       "required": ["path", "content"]}}},
    {"type": "function", "function": {
        "name": "run_terminal_command", "description": "Execute a bash/Linux command in the environment and get output.",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}},
                       "required": ["command"]}}}
]


## การเชื่อมโยงเครื่องมือ (Tool Mapping)
ส่วนนี้เป็นการสร้างดิกชันนารี `FUNCTIONS` เพื่อจับคู่ชื่อเครื่องมือที่เป็นข้อความ (String) กับฟังก์ชันของ Python จริงๆ และสร้างฟังก์ชัน `execute` เพื่อเรียกใช้งานเครื่องมือนั้นๆ พร้อมส่งอาร์กิวเมนต์ที่แปลงมาจาก JSON


In [ ]:
# Mapping of tool names to their respective functions
FUNCTIONS = {
    "calc": calc,
    "read_file": read_file,
    "write_file": write_file,
    "run_terminal_command": run_terminal_command
}


def execute(name: str, arguments: str) -> str:
    """Execute a specified tool with arguments."""
    try:
        return FUNCTIONS[name](**json.loads(arguments or "{}"))
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

## ส่วนที่ 2: การสนทนาและการเรียกใช้งาน API
ในส่วนนี้เรากำหนดค่าเริ่มต้นในการเชื่อมต่อกับ Groq API (เช่น URL และ Model) และสร้างฟังก์ชัน `chat` เพื่อส่งประวัติการสนทนาพร้อมรายการเครื่องมือ (Tools) ไปยังโมเดล AI และรับผลลัพธ์กลับมา


In [ ]:
"""Step 2 (Split): Part 2 - Chat and Runtime execution"""
import os
import sys
import httpx
from google.colab import userdata


URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

try:
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = os.environ.get('GROQ_API_KEY')

def chat(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 800}
    if tools:
        body["tools"] = tools

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]



## ส่วนที่ 3: วงจรการทำงานของเอเจนต์ (Agent Loop)
ฟังก์ชัน `run_agent` จะทำหน้าที่ควบคุมขั้นตอนการทำงาน โดยจะถามโมเดล AI หากโมเดลต้องการเรียกใช้เครื่องมือ (Tool Calls) เอเจนต์จะรันเครื่องมือนั้นและส่งผลลัพธ์กลับไปให้โมเดลประมวลผลต่อ ทำซ้ำแบบนี้เรื่อยๆ จนกว่าจะได้คำตอบสุดท้าย


In [ ]:
"""Step 3 (new file): the agent loop. Call the model, run requested tools, repeat until it answers."""
def run_agent(messages: list[dict], max_steps: int = 8) -> str:
    for _ in range(max_steps):  # max_steps prevents infinite loops
        reply = chat(messages, SCHEMAS)
        calls = reply.get("tool_calls") or []
        messages.append({"role": "assistant", "content": reply.get("content") or "",
                         **({"tool_calls": calls} if calls else {})})
        if not calls:
            return messages[-1]["content"]
        for call in calls:
            fn = call["function"]
            print(f"[tool] {fn['name']}({fn['arguments']})")
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": execute(fn["name"], fn["arguments"])})
    return f"[stopped after {max_steps} steps without a final answer]"


## การทดสอบการทำงานเบื้องต้น (Testing the Agent)
โค้ดส่วนนี้เป็นการทดสอบส่งคำถามทักทายทั่วไป เพื่อดูว่าเอเจนต์สามารถตอบคำถามได้ถูกต้องโดยไม่ต้องเรียกใช้เครื่องมือหรือไม่


In [ ]:
question =  "hello how are you?"
print(run_agent([{"role": "user", "content": question}]))

I'm doing well, thank you for asking! How can I help you today?



## ทดสอบการคำนวณคณิตศาสตร์ (Math Tool Test)
ทดสอบส่งโจทย์เลขเพื่อให้เอเจนต์ตัดสินใจเรียกใช้งานเครื่องมือ `calc` เพื่อคำนวณผลลัพธ์


In [ ]:
question =  "123+68"
print(run_agent([{"role": "user", "content": question}]))

[tool] calc({"expression":"123+68"})
123 + 68 = 191



## ทดสอบการอ่านไฟล์ (File Tool Test)
ทดสอบถามเกี่ยวกับเนื้อหาในไฟล์ ซึ่งเอเจนต์จะพยายามเรียกใช้เครื่องมือ `read_file` เพื่อตรวจสอบไฟล์ในไดเรกทอรี


In [ ]:
question =  "what inside note.txt"
print(run_agent([{"role": "user", "content": question}]))

[tool] read_file({"path":"note.txt"})
The file `note.txt` doesn't exist in the working directory — reading it returned a "No such file or directory" error.

A few possibilities:
- The file might be in a different directory or have a different name.
- The file may not have been uploaded/created yet.

If you can tell me the exact path or filename (e.g. `./notes/note.txt`, `my-note.txt`, `NOTE.TXT`), I'll take another look. You could also paste the contents directly if you just need help with the file's content.



## คลาสเซสชันการสนทนา (Chat Session Class)
สร้างคลาส `ChatSession` เพื่อเก็บประวัติการคุยทั้งหมด (Conversation History) ของผู้ใช้ เพื่อให้เอเจนต์สามารถจดจำบริบทของการสนทนาก่อนหน้านี้ได้


In [ ]:
class ChatSession:
    def __init__(self):
        # Keep track of the full conversation history
        self.messages = []

    def send_message(self, user_message: str) -> str:
        # 1. Append the new user query to the history
        self.messages.append({"role": "user", "content": user_message})

        # 2. Run the agent loop starting with the current history
        # Note: run_agent will append assistant replies and tool results to self.messages in place
        response = run_agent(self.messages)
        return response


## หน้าต่างแชทจำลอง (Interactive Chat Widget UI)
ส่วนสุดท้ายคือการสร้างหน้าต่างแชทโต้ตอบ (UI) สวยงามโดยใช้ `ipywidgets` เพื่อให้ผู้ใช้สามารถพิมพ์คุยกับเอเจนต์ได้แบบเรียลไทม์ใน Notebook


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Initialize a fresh chat session
ui_session = ChatSession()

# HTML template for chat messages
def format_message(sender, message):
    bg_color = "#E3F2FD" if sender == "You" else "#FFEBEE" if sender == "System Error" else "#F5F5F5"
    align = "right" if sender == "You" else "left"
    margin = "margin-left: 20%;" if sender == "You" else "margin-right: 20%;"
    text_color = "color: #C62828;" if sender == "System Error" else ""
    return f'''
    <div style="text-align: {align}; margin: 5px 0;">
        <div style="display: inline-block; background-color: {bg_color}; padding: 8px 12px;
                    border-radius: 12px; max-width: 80%; text-align: left; {margin} {text_color}">
            <strong>{sender}:</strong> {message.replace('\n', '<br>')}
        </div>
    </div>
    '''

# UI Elements - Fixed overflow property for correct scrolling
output_area = widgets.Output(layout=widgets.Layout(height='300px', overflow='auto', border='1px solid #ccc', padding='10px'))
text_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='18%'))

# Action on sending a message
def on_send_click(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return

    # Clear input immediately for better UX
    text_input.value = ''

    # Append user message to UI
    with output_area:
        display(HTML(format_message("You", user_msg)))

    # Get agent response safely
    try:
        agent_reply = ui_session.send_message(user_msg)
        # Append agent reply to UI
        with output_area:
            display(HTML(format_message("Agent", agent_reply)))
    except Exception as e:
        with output_area:
            error_msg = str(e)
            if "429" in error_msg:
                error_msg = "Rate limit reached (HTTP 429). Please wait a brief moment and try again."
            display(HTML(format_message("System Error", error_msg)))

send_button.on_click(on_send_click)
text_input.on_submit(on_send_click) # Send when pressing Enter

# Display the Layout
print("Interactive Chat Widget:")
display(widgets.VBox([output_area, widgets.HBox([text_input, send_button])]))

Interactive Chat Widget:
